In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

import warnings
warnings.filterwarnings("ignore")


In [ ]:
data = pd.read_csv("/content/ACLED Data_2026-03-08.csv")


In [ ]:
data = data[["event_id_cnty", "country", "event_date", "disorder_type", "event_type",
             "interaction", "civilian_targeting", "region", "notes",
             "fatalities"]]
data


In [ ]:
data["event_date"] = pd.to_datetime(data["event_date"])
data = data.sort_values(by=["country", "event_date"]).reset_index(drop=True)


In [ ]:
# Drop columns with too many NaN values or not useful for modelling
data = data.drop(columns=["civilian_targeting", "event_id_cnty", "region", "disorder_type"])
data


In [ ]:
df = pd.get_dummies(data, columns=["event_type"], dtype=int)
df


In [ ]:
df[["actor1", "actor2"]] = df["interaction"].str.split("-", expand=True)
df["actor1"] = df["actor1"].str.replace(" only", "", regex=False)
df["actor2"] = df["actor2"].str.replace(" only", "", regex=False)


In [ ]:
actor1_dummies = pd.get_dummies(df["actor1"], prefix="actor", dtype=int)
actor2_dummies = pd.get_dummies(df["actor2"], prefix="actor", dtype=int)
actor_features = actor1_dummies.add(actor2_dummies, fill_value=0)

df = pd.concat([df, actor_features], axis=1)
df = df.drop(["interaction", "actor1", "actor2"], axis=1)
df


## BERT Embeddings for Event Notes
BERT embeddings are computed per event (row) on the `notes` column before weekly aggregation.
This is safe — each event's text is embedded independently; no future information is used.


In [ ]:
# !pip install sentence-transformers  # Uncomment if not installed

from sentence_transformers import SentenceTransformer

bert_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = bert_model.encode(
    df["notes"].fillna("").tolist(),
    batch_size=64,
    show_progress_bar=True
)

embeddings_df = pd.DataFrame(
    embeddings,
    columns=[f"bert_{i}" for i in range(embeddings.shape[1])]
)

df = pd.concat([df.reset_index(drop=True), embeddings_df], axis=1)
df = df.drop(columns=["notes"])  # notes no longer needed; captured in embeddings
df


## Weekly Aggregation
Group events by (country, week) to get one row per country per week.
- Fatalities: summed
- Actor / event-type one-hot columns: summed (counts per week)
- BERT embeddings: averaged across events in the week


In [ ]:
df["week"] = df["event_date"].dt.to_period("W")

bert_cols        = [c for c in df.columns if c.startswith("bert_")]
actor_cols       = [c for c in df.columns if c.startswith("actor_")]
event_type_cols  = [c for c in df.columns if c.startswith("event_type_")]

weekly_df = df.groupby(["country", "week"]).agg(
    {
        "fatalities": "sum",
        **{col: "sum"  for col in actor_cols},
        **{col: "sum"  for col in event_type_cols},
        **{col: "mean" for col in bert_cols},
    }
).reset_index()

weekly_df


## Feature Engineering
All engineered features are derived from *current or past* data only — no future information is used here.


In [ ]:
# Total event count per week
weekly_df["event_count"] = weekly_df[event_type_cols].sum(axis=1)

# Violent event sub-total and ratio
weekly_df["violent_events"] = (
    weekly_df["event_type_Battles"] +
    weekly_df["event_type_Explosions/Remote violence"] +
    weekly_df["event_type_Violence against civilians"]
)
weekly_df["violent_event_ratio"] = (
    weekly_df["violent_events"] / weekly_df["event_count"].replace(0, 1)
)

# State-actor ratio
weekly_df["state_actor_ratio"] = (
    weekly_df["actor_State forces"] /
    weekly_df[[
        "actor_Civilians", "actor_External/Other forces",
        "actor_Identity militia", "actor_Political militia",
        "actor_Protesters", "actor_Rebel group",
        "actor_Rioters", "actor_State forces"
    ]].sum(axis=1).replace(0, 1)
)

# Event diversity (number of distinct event types in the week)
weekly_df["event_diversity"] = (weekly_df[event_type_cols] > 0).sum(axis=1)

weekly_df.head()


## Lag Features  *(data-leakage fix)*
**Bug fixed:** The original code used `.rolling(N)` which included the *current* week in the window,
so `fatalities_lag2` at week *t* was actually `fatalities[t-1] + fatalities[t]`.
That is not a lag — it leaks current-week data into the feature.

**Fix:** Sort by week, then use `.shift(1).rolling(N)` so that at week *t*:
- `fatalities_lag2` = sum of weeks *t-1* and *t-2*
- `fatalities_lag4` = sum of weeks *t-1* through *t-4*


In [ ]:
# Sort within each country by week before computing rolling windows
weekly_df = weekly_df.sort_values(["country", "week"]).reset_index(drop=True)

# Shift by 1 FIRST so that week-t only sees history up to week t-1
weekly_df["fatalities_lag2"] = (
    weekly_df.groupby("country")["fatalities"]
    .transform(lambda s: s.shift(1).rolling(2, min_periods=1).sum())
)

weekly_df["fatalities_lag4"] = (
    weekly_df.groupby("country")["fatalities"]
    .transform(lambda s: s.shift(1).rolling(4, min_periods=1).sum())
)

weekly_df[["country", "week", "fatalities", "fatalities_lag2", "fatalities_lag4"]].head(10)


## Target Variable: Conflict Escalation  *(data-leakage fix)*
`future_fatalities` is computed as the sum of fatalities over the *next 4 weeks*.
`conflict_escalation = 1` if future fatalities > 2× current week's fatalities.

**Bug fixed (original code):**
- `future_fatalities` was left as a column in `X`, directly encoding the label → **severe leakage**.
- `fillna(0)` was applied *after* training, so NaN rows (last 4 weeks per country,
  where future is unknown) were silently included with a NaN label → **corrupted labels**.

**Fix:**
- `future_fatalities` is used only to compute the label, then **dropped**.
- Rows where `conflict_escalation` is NaN (last 4 weeks per country, no future) are **dropped before splitting**.


In [ ]:
# future_fatalities: used ONLY to derive the label, then dropped
weekly_df["future_fatalities"] = (
    weekly_df.groupby("country")["fatalities"].shift(-1) +
    weekly_df.groupby("country")["fatalities"].shift(-2) +
    weekly_df.groupby("country")["fatalities"].shift(-3) +
    weekly_df.groupby("country")["fatalities"].shift(-4)
)

weekly_df["conflict_escalation"] = (
    weekly_df["future_fatalities"] > 2 * weekly_df["fatalities"]
).astype("Int64")   # Int64 preserves NaN for rows that have no future window

# Drop future_fatalities — it must NEVER enter the feature matrix
weekly_df = weekly_df.drop(columns=["future_fatalities"])

# Drop rows where the label is NaN (last 4 weeks per country — future is unknown)
weekly_df = weekly_df.dropna(subset=["conflict_escalation"]).reset_index(drop=True)
weekly_df["conflict_escalation"] = weekly_df["conflict_escalation"].astype(int)

print("Label distribution:")
print(weekly_df["conflict_escalation"].value_counts())
weekly_df.head()


## Temporal Train / Test Split  *(data-leakage fix)*
**Bug fixed:** `train_test_split(..., shuffle=False)` split rows in their stored order.
Because the DataFrame is sorted by *country first then week*, the "test" rows end up being
the last countries alphabetically — not the most recent time period. This mixes future and
past data across the split boundary.

**Fix:** Split on the **time axis** — train on weeks before a cutoff, test on weeks after.
This mirrors real-world use where the model is trained on historical data and evaluated on future data.


In [ ]:
from sklearn.model_selection import train_test_split

# Convert Period to timestamp for comparison
weekly_df["week_ts"] = weekly_df["week"].dt.to_timestamp()

# Use 80th percentile of weeks as the cutoff (80 / 20 time split)
cutoff = weekly_df["week_ts"].quantile(0.80)
print(f"Train: weeks up to {cutoff.date()}   |   Test: weeks after {cutoff.date()}")

train_df = weekly_df[weekly_df["week_ts"] <= cutoff].copy()
test_df  = weekly_df[weekly_df["week_ts"] >  cutoff].copy()

print(f"Train size: {len(train_df)}   Test size: {len(test_df)}")

# Drop non-feature columns from X
drop_cols = ["conflict_escalation", "country", "week", "week_ts"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["conflict_escalation"]

X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df["conflict_escalation"]

print("\nFeature matrix shape — Train:", X_train.shape, "  Test:", X_test.shape)


## Fill Remaining NaNs
The only remaining NaNs are in `fatalities_lag2` / `fatalities_lag4` for the first few weeks of
each country (not enough history yet). We fill those with 0 *separately on train and test*
to avoid any cross-contamination.


In [ ]:
# Fill NaNs only in lag features; do it separately on train and test
lag_cols = ["fatalities_lag2", "fatalities_lag4"]
X_train[lag_cols] = X_train[lag_cols].fillna(0)
X_test[lag_cols]  = X_test[lag_cols].fillna(0)

# Sanity check — no NaNs should remain
assert X_train.isnull().sum().sum() == 0, "NaNs remain in X_train!"
assert X_test.isnull().sum().sum()  == 0, "NaNs remain in X_test!"
print("No NaNs remaining. Ready for modelling.")


## Model Training — XGBoost Classifier


In [ ]:
from xgboost import XGBClassifier

clf = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)

clf.fit(X_train, y_train)
print("Model trained.")


## Evaluation


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")


## Conflict Risk Index
Scale predicted probabilities to a 0-100 risk index.


In [ ]:
test_df = test_df.copy()
test_df["risk_index"] = y_proba * 100

# Top-10 highest-risk country-weeks
test_df[["country", "week", "fatalities", "conflict_escalation", "risk_index"]] \
    .sort_values("risk_index", ascending=False) \
    .head(10)


In [ ]:
import matplotlib.pyplot as plt

feat_imp = pd.Series(clf.feature_importances_, index=X_train.columns)
top20 = feat_imp.nlargest(20)

plt.figure(figsize=(10, 6))
top20.sort_values().plot(kind="barh")
plt.title("Top-20 Feature Importances")
plt.tight_layout()
plt.show()


In [ ]:
# 1. TOP COUNTRIES BY FATALITIES (Bar Chart)
# ─────────────────────────────────────────────
top_countries = (
    weekly_df.groupby("country")["fatalities"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, palette="Reds_r")
plt.title("Top 15 Countries by Total Fatalities", fontsize=14, fontweight="bold")
plt.xlabel("Total Fatalities")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

In [ ]:
# 4. WHAT DRIVES ESCALATION? — SIMPLE GROUPED BAR
# Average feature values for escalated vs non-escalated weeks
# ─────────────────────────────────────────────
compare_cols = [
    "fatalities", "violent_events", "event_count",
    "event_diversity", "state_actor_ratio"
]

grouped = (
    weekly_df.groupby("conflict_escalation")[compare_cols]
    .mean()
    .T
    .rename(columns={0: "No Escalation", 1: "Escalation"})
)

# Normalise to 0-1 for fair comparison across different scales
grouped_norm = grouped.div(grouped.max(axis=1), axis=0)

grouped_norm.plot(
    kind="bar",
    figsize=(10, 5),
    color=["steelblue", "crimson"],
    edgecolor="white",
    width=0.6
)
plt.title("What Looks Different in Escalation Weeks?\n(values normalised for comparison)",
          fontsize=13, fontweight="bold")
plt.ylabel("Normalised Average Value")
plt.xticks(rotation=25, ha="right")
plt.legend(title="")
plt.tight_layout()
plt.show()


In [ ]:
# 2. FATALITIES vs ESCALATION RATE BUBBLE CHART
# Big bubbles = more conflict weeks on record
# ─────────────────────────────────────────────
country_stats = weekly_df.groupby("country").agg(
    total_fatalities=("fatalities", "sum"),
    escalation_rate=("conflict_escalation", "mean"),
    weeks=("conflict_escalation", "count")
).reset_index()

# Focus on countries with enough data
country_stats = country_stats[country_stats["weeks"] >= 10]

plt.figure(figsize=(12, 7))
scatter = plt.scatter(
    country_stats["total_fatalities"],
    country_stats["escalation_rate"] * 100,
    s=country_stats["weeks"] * 3,
    c=country_stats["escalation_rate"],
    cmap="RdYlGn_r",
    alpha=0.7,
    edgecolors="gray",
    linewidths=0.5
)

# Label the most extreme countries
top = country_stats.nlargest(8, "total_fatalities")
for _, row in top.iterrows():
    plt.annotate(
        row["country"],
        (row["total_fatalities"], row["escalation_rate"] * 100),
        fontsize=7.5, ha="left",
        xytext=(5, 3), textcoords="offset points"
    )

plt.colorbar(scatter, label="Escalation Rate")
plt.xscale("log")
plt.xlabel("Total Fatalities (log scale)")
plt.ylabel("Escalation Rate (%)")
plt.title("Fatalities vs Escalation Rate\n(bubble size = number of weeks observed)",
          fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()